<a href="https://colab.research.google.com/github/hadi-hosseini/bandit/blob/main/Bandit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import math
import cvxpy as cp
import numpy as np
from scipy.stats import ortho_group
from tqdm import tqdm

# Parameters
k = 3
d = 1000
n_samples = 1000
sigma = 1.0
T = 500

orthogonal_matrix = ortho_group.rvs(dim=d)
mu = orthogonal_matrix[:k]
print(mu)

[[ 0.02847803  0.04214837 -0.04093468 ...  0.0422777   0.00405653
   0.03456312]
 [-0.06156817 -0.0192492  -0.03927995 ...  0.00948667  0.04147329
   0.10573864]
 [-0.00148101 -0.00312553 -0.02706315 ...  0.05147368 -0.00928661
   0.03007578]]


In [27]:
# Verify orthogonality
for i in range(k):
    for j in range(i+1, k):
        print(f"Dot product μ_{i+1}·μ_{j+1}: {np.dot(mu[i], mu[j])}")

Dot product μ_1·μ_2: -4.640385298237959e-17
Dot product μ_1·μ_3: -3.5344990823027445e-17
Dot product μ_2·μ_3: 8.977193988179977e-17


In [28]:
# create offline dataset
def create_logged_data(k, d, n_samples, sigma, mu):
  samples = []
  for i in range(k):
    # Generate samples from N(μ_i, σ²I)
    samples.append(np.random.normal(loc=mu[i], scale=sigma, size=(n_samples, d)))
  return samples

logged_data = create_logged_data(k, d, n_samples, sigma, mu)

In [29]:
# implement UCB algorithm
class UCBAlgorithm:
    def __init__(self, k, d, true_means, logged_data, perturbation):
        self.k = k
        self.d = d
        self.true_means = true_means
        self.logged_data = logged_data

        self.N = np.zeros(k)
        self.total_rewards = np.zeros(k)
        self.empirical_rewards = np.zeros(k)
        self.empirical_means = np.zeros((k, d))
        self.perturbation = perturbation

    def get_reward(self, x):
        return np.dot(self.true_means[0] + self.perturbation, x)

    def select_arm(self, t):
        if t < self.k:
            return t

        ucb_values = np.zeros(self.k)
        for j in range(self.k):
            mean_term = self.empirical_rewards[j]
            confidence_bound = math.sqrt(2 * math.log(t) / self.N[j])

            ucb_values[j] = mean_term + confidence_bound

        return np.argmax(ucb_values)

    def update(self, arm, reward):
        self.N[arm] += 1
        self.total_rewards[arm] += reward
        self.empirical_rewards[arm] = self.total_rewards[arm] / self.N[arm]

    def run(self, T):
        rewards = np.zeros(T)
        chosen_arms = np.zeros(T, dtype=int)

        for t in range(T):
            arm = self.select_arm(t)
            sample = self.logged_data[arm][int(self.N[arm])]
            reward = self.get_reward(sample)

            self.update(arm, reward)
            self.empirical_means[arm] = self.empirical_means[arm] + (sample - self.empirical_means[arm])/self.N[arm]

            rewards[t] = reward
            chosen_arms[t] = arm

        return rewards, chosen_arms

In [30]:
# run UCB without perturbation
perturbation = 0.0
ucb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb.run(T)
print("\nNumber of pulls per arm:", ucb.N)
print(chosen_arms)


Number of pulls per arm: [488.   7.   5.]
[0 1 2 0 2 0 2 1 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 2 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 

In [31]:
# find adversary perturbation
# should learn this perturbation based on the logged data

class FindPerturbation:
    def __init__(self, k, d, true_means, logged_data, epsilon):
        self.k = k
        self.d = d
        self.true_means = true_means
        self.logged_data = logged_data
        self.epsilon = epsilon

        self.N = np.zeros(k)
        self.empirical_means = np.zeros((k, d))
        self.constraints = []
        self.perturbation = None
        self.history = []
        self.turn = 1

    def select_arm(self, t):
        if t < self.k:
            return t

        turn = self.turn
        # self.turn += 1
        # if self.turn == k:
        #   self.turn = 1
        return turn

    def find_perturbation_with_l2_ball(self, arm, t):
        x = cp.Variable(self.d)

        for j in range(self.k):
            if j != arm:
              d_j = self.empirical_means[arm] - self.empirical_means[j]
              c_j = (math.sqrt(2 * math.log(t) / self.N[j]) - math.sqrt(2 * math.log(t) / self.N[arm])) - np.dot(self.true_means[0], d_j)
              self.constraints.append(x @ d_j >= c_j + 1e-6)

        constraints = self.constraints[:]
        constraints.append(cp.norm(x, 2) <= self.epsilon)
        prob = cp.Problem(cp.Minimize(0), constraints)
        prob.solve()

        if prob.status == 'optimal':
          self.history.append(x.value)
          return x.value
        else:
          return None


    def run(self, T):
        chosen_arms = np.zeros(T, dtype=int)

        for t in tqdm(range(T)):
            arm = self.select_arm(t)

            if t >= self.k:
              perturbation = self.find_perturbation_with_l2_ball(arm, t)

              if perturbation is None:
                return chosen_arms

              self.perturbation = perturbation

            sample = self.logged_data[arm][int(self.N[arm])]
            self.N[arm] += 1
            self.empirical_means[arm] = self.empirical_means[arm] + (sample - self.empirical_means[arm])/self.N[arm]

            chosen_arms[t] = arm

        return chosen_arms

In [ ]:
epsilon = 0.5
find_perturbation = FindPerturbation(k, d, mu, logged_data, epsilon)
chosen_arms = find_perturbation.run(T)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

 71%|███████   | 356/500 [11:52<10:38,  4.43s/it]

In [ ]:
perturbation = find_perturbation.perturbation
norm = np.linalg.norm(perturbation)
print(norm)

In [ ]:
history = find_perturbation.history

for hist in history:
  ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, hist)
  rewards, chosen_arms = ucb_with_perturb.run(T)
  print("\nNumber of pulls per arm:", ucb_with_perturb.N)
  print(chosen_arms)

In [ ]:
# run UCB with perturbation
ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb_with_perturb.run(T)
print("\nNumber of pulls per arm:", ucb_with_perturb.N)
print(chosen_arms)